# Scrape

## Setup Selenium

In [1]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

print("🚀 กำลังเปิดเว็บ กกต. และเลื่อนหน้าจอ...")
driver.get("https://party.ect.go.th/party-info")
time.sleep(3)

# เลื่อนจอลงไปเรื่อยๆ จนสุด
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2) 
    
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        print("✅ เลื่อนจนสุดหน้าเว็บแล้ว!")
        break
    last_height = new_height

soup = BeautifulSoup(driver.page_source, 'html.parser')
driver.quit() 

🚀 กำลังเปิดเว็บ กกต. และเลื่อนหน้าจอ...
✅ เลื่อนจนสุดหน้าเว็บแล้ว!


## Find Data

In [2]:
cards = soup.find_all('a', href=lambda x: x and '/party-info/' in x)

party_endpoints = [] 

for card in cards:
    endpoint_href = card.get('href')
    
    th_name_tag = card.find('div', class_=lambda c: c and 'text-xl' in c)
    th_name = th_name_tag.text.strip() if th_name_tag else ''
    
    party_id_tag = card.find('span', class_=lambda c: c and 'text-primary' in c)
    party_id = party_id_tag.text.strip() if party_id_tag else ''
    
    if th_name and 'ข้อมูล' not in th_name and 'สร้างสรรค์' not in th_name:
        party_endpoints.append({
            'ID': party_id,
            'Name_TH': th_name,
            'href': endpoint_href  
        })

df = pd.DataFrame(party_endpoints)
df = df.drop_duplicates(subset=['Name_TH']).reset_index(drop=True)

party_endpoints_clean = df.to_dict('records')

print(f"\n🎉 สำเร็จ! ดึง Endpoint มาได้ทั้งหมด {len(party_endpoints_clean)} พรรค")
print("ตัวอย่าง 3 พรรคแรก (พร้อม href):")
print(party_endpoints_clean[:3])


🎉 สำเร็จ! ดึง Endpoint มาได้ทั้งหมด 75 พรรค
ตัวอย่าง 3 พรรคแรก (พร้อม href):
[{'ID': '001', 'Name_TH': 'ประชาธิปัตย์', 'href': '/party-info/1'}, {'ID': '002', 'Name_TH': 'ประชากรไทย', 'href': '/party-info/2'}, {'ID': '013', 'Name_TH': 'ความหวังใหม่', 'href': '/party-info/6'}]


## Save Constants
for reusable

In [3]:
import json

data_list = party_endpoints_clean 

party_ids = [item['ID'] for item in data_list if item.get('ID')]
party_names_th = [item['Name_TH'] for item in data_list if item.get('Name_TH')]


party_dict = {
    item['ID']: {
        'th': item['Name_TH'], 
        'href': item.get('href', '') 
    } 
    for item in data_list if item.get('ID')
}

with open('./constants.py', 'w', encoding='utf-8') as f:
    f.write("# รายชื่อรหัสพรรคทั้งหมด (List)\n")
    f.write(f"PARTY_IDS = {party_ids}\n\n")
    
    f.write("# ชื่อพรรคภาษาไทย (List)\n")
    f.write(f"PARTY_NAMES_TH = {party_names_th}\n\n")
    
    f.write("# Dictionary ค้นหาข้อมูลพรรคจากรหัส\n")
    dict_str = json.dumps(party_dict, ensure_ascii=False, indent=4)
    f.write(f"PARTY_DICT = {dict_str}\n")

print("✅ สร้างไฟล์ constants.py สำเร็จเรียบร้อย พร้อมเอาไปใช้งานต่อ!")

✅ สร้างไฟล์ constants.py สำเร็จเรียบร้อย พร้อมเอาไปใช้งานต่อ!


In [ ]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver

from constants import PARTY_DICT

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

results = []

# 🌟 แก้ไขบรรทัดนี้: ใช้ .items() เพื่อดึงทั้ง รหัส (party_id) และ ข้อมูล (party_info)
for party_id, party_info in PARTY_DICT.items():
    
    # ดึง href และ th ออกมาจาก party_info
    target_url = "https://party.ect.go.th" + party_info["href"]
    print(f"กำลังดึงข้อมูล: {party_info['th']}...")
    
    driver.get(target_url)
    time.sleep(2) 
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    def get_number_by_label(soup, keyword):
        label_text = soup.find(string=re.compile(keyword))
        if label_text:
            number_div = label_text.find_next('div', class_=lambda c: c and 'text-[40px]' in c)
            if number_div:
                return number_div.text.strip().replace(',', '')
        return "0" 
    
    branch_count = get_number_by_label(soup, "สาขา")
    rep_count = get_number_by_label(soup, "ตัวแทน")
    member_count = get_number_by_label(soup, "สมาชิก")
    
    results.append({
        "ID": party_id, # ใช้รหัสพรรคจาก Key
        "Name": party_info["th"], # ชื่อไทยเก็บอยู่ในคีย์ 'th'
        "Branch": int(branch_count) if branch_count.isdigit() else 0,
        "Representative": int(rep_count) if rep_count.isdigit() else 0,
        "Member": int(member_count) if member_count.isdigit() else 0
    })

driver.quit()

df = pd.DataFrame(results)
print("\n🎉 สกัดข้อมูลสำเร็จ:")
print(df)

df.to_csv('./data/ect_party_stats.csv', index=False, encoding='utf-8-sig')

กำลังดึงข้อมูล: ประชาธิปัตย์...
กำลังดึงข้อมูล: ประชากรไทย...
กำลังดึงข้อมูล: ความหวังใหม่...
กำลังดึงข้อมูล: เครือข่ายชาวนาแห่งประเทศไทย...
กำลังดึงข้อมูล: เพื่อไทย...
กำลังดึงข้อมูล: ชาติพัฒนา...
กำลังดึงข้อมูล: ชาติไทยพัฒนา...
กำลังดึงข้อมูล: อนาคตไทย...
กำลังดึงข้อมูล: ภูมิใจไทย...
กำลังดึงข้อมูล: สังคมประชาธิปไตยไทย...
กำลังดึงข้อมูล: รักชาติ...
กำลังดึงข้อมูล: ประชาธิปไตยใหม่...
กำลังดึงข้อมูล: พลังบูรพา...
กำลังดึงข้อมูล: ครูไทยเพื่อประชาชน...
กำลังดึงข้อมูล: พลังท้องถิ่นไท...
กำลังดึงข้อมูล: ประชาชน...
กำลังดึงข้อมูล: ไทยก้าวใหม่...
กำลังดึงข้อมูล: เสรีรวมไทย...
กำลังดึงข้อมูล: รักษ์ธรรม...
กำลังดึงข้อมูล: พลังประชาธิปไตย...
กำลังดึงข้อมูล: พลังสุราษฎร์...
กำลังดึงข้อมูล: พลังไทยรักชาติ...
กำลังดึงข้อมูล: เพื่อชีวิตใหม่...
กำลังดึงข้อมูล: ทางเลือกใหม่...
กำลังดึงข้อมูล: เศรษฐกิจ...
กำลังดึงข้อมูล: สร้างอนาคตไทย...
กำลังดึงข้อมูล: พลังธรรมใหม่...
กำลังดึงข้อมูล: ไทยธรรม...
กำลังดึงข้อมูล: รวมพลัง...
กำลังดึงข้อมูล: ไทยพร้อม...
กำลังดึงข้อมูล: ปวงชนไทย...
กำลังดึงข้อมูล: เพื่อชาต